# Week 1 Final Project · 学生成绩分析与预测

这是一份可从头独立运行的参考项目。建议先完成各天作业，再对照本例完成自己的项目。
项目问题：在第二阶段成绩已知的假设下，用 G1/G2 及少量学习信息估计 G3。
数据：UCI 数学课程 395 条记录，成绩 0～20。来源、许可和字段解释见 ../data/README.md。
数据缺少可用时间戳，因此这是回顾性教学评估，不证明可以直接上线预警。

流程：读取与审计 → 固定拆分 → 训练集清洗和探索 → 验证集选模 → 冻结 → 测试与误差分析。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "assignments/bootcamp/week1/data/raw/student-mat.csv").exists())
raw = pd.read_csv(ROOT / "assignments/bootcamp/week1/data/raw/student-mat.csv", sep=";")
# 固定划分，第三至七天始终使用相同学生；实际行数为 276/59/60。
train, heldout = train_test_split(raw, test_size=0.30, random_state=42)
val, test = train_test_split(heldout, test_size=0.50, random_state=42)
train, val, test = train.copy(), val.copy(), test.copy()
print("训练/验证/测试：", len(train), len(val), len(test))


### 1. 数据质量与清洗决策

原始数据无缺失；先检查，不凭空添加清洗。
对原始全表只检查 schema/范围等预定质量规则，不用测试分布决定特征。
本例目标有效，无需删除；不把 G3=0 当缺失，不把重复特征当作重复学生。
缺失特征填补在 Pipeline 内完成。学习清洗方法时另用训练集教学副本。

In [ ]:
assert raw.shape == (395,33)
assert raw["G3"].between(0,20).all()
quality = pd.DataFrame({"dtype":train.dtypes.astype(str),"missing":train.isna().sum()})
display(quality)
print("训练集完全重复行：", train.duplicated().sum())
print("训练集有效零分：", int((train["G3"]==0).sum()))
# 原始数据没有 ID，因此不根据相似特征擅自删人。
assert not train.isna().any().any()

### 2. 清洗案例：教学副本

以下副本仅演示清洗，不能与真实建模数据混合。保留原始数据来源和修改过程。

In [ ]:
# 错误仅注入训练集的副本，不改动原始文件。
dirty = train.head(12)[["school", "age", "studytime", "G1", "G2", "G3"]].copy()
dirty.insert(0, "record_id", dirty.index)
dirty = dirty.reset_index(drop=True)
dirty.loc[0, "age"] = np.nan
dirty.loc[1, "G2"] = np.nan
dirty["studytime"] = dirty["studytime"].astype("str")
dirty.loc[2, "studytime"] = "unknown"
dirty.loc[3, "G3"] = 99
dirty.loc[4, "school"] = " gp "
dirty.loc[5, "G3"] = np.nan
dirty = pd.concat([dirty, dirty.iloc[[6]]], ignore_index=True)
display(dirty)


In [ ]:
def clean_students(frame, medians):
    result = frame.copy()
    result["school"] = result["school"].str.strip().str.upper()
    for col in ["age", "studytime", "G1", "G2", "G3"]:
        result[col] = pd.to_numeric(result[col], errors="coerce")
    result.loc[~result["G3"].between(0, 20), "G3"] = np.nan
    # 此例的重复 ID 已知来自复制；真实数据必须先确认记录身份。
    result = result.drop_duplicates("record_id").dropna(subset=["G3"])
    result[medians.index] = result[medians.index].fillna(medians)
    return result.reset_index(drop=True)
medians = train[["age", "studytime", "G1", "G2"]].median()
clean = clean_students(dirty, medians)
assert len(clean) == 10
assert clean["record_id"].is_unique
assert clean["G3"].between(0, 20).all()
assert not clean.isna().any().any()
display(clean)


### 3. 训练集可视化

观察分布、分组差异和关系；相关性不作因果解释。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(data=train, x="G3", bins=np.arange(-0.5,21.5,1), ax=axes[0,0])
axes[0,0].set_title("Training final grades (0-20)")
sns.boxplot(data=train, x="studytime", y="G3", ax=axes[0,1])
axes[0,1].set_title("Study-time categories vs grade")
sns.scatterplot(data=train, x="G2", y="G3", alpha=0.5, ax=axes[1,0])
axes[1,0].set_title("Second-period vs final grade")
sns.heatmap(train[["G1","G2","G3","studytime","failures"]].corr(),
            annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1,1])
axes[1,1].set_title("Pearson correlation (training only)")
fig.tight_layout()
plt.show()


In [ ]:
display(train.groupby("studytime")["G3"].agg(["count","mean","median"]))
print("训练集 G2/G3 相关系数：", train["G2"].corr(train["G3"]))

### 4. 特征、基准和回归

主线候选仅 G2 或四特征。两者均使用同一训练/验证记录和同一 MAE 选择标准。
school、internet、Ridge 是第六天拓展练习，不参与本参考项目的候选集合。
Pipeline 保证填补统计量从训练数据学习。

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
features = ["G1", "G2", "studytime", "failures"]
target = "G3"
X_train, y_train = train[features], train[target]
X_val, y_val = val[features], val[target]
def make_model():
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("regressor", LinearRegression())])
def metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "MSE": mean_squared_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
model = make_model().fit(X_train, y_train)
val_pred = model.predict(X_val)
display(pd.DataFrame({
    "baseline": metrics(y_val, baseline.predict(X_val)),
    "linear": metrics(y_val, val_pred)}).T)


In [ ]:
# 两个预先规定的候选方案；选择标准仅为验证集 MAE。
feature_sets = {"G2_only": ["G2"], "multi": ["G1","G2","studytime","failures"]}
candidates = {}
rows = []
for name, cols in feature_sets.items():
    fitted = make_model().fit(train[cols], train["G3"])
    candidates[name] = fitted
    rows.append({"model": name, **metrics(val["G3"], fitted.predict(val[cols]))})
validation_results = pd.DataFrame(rows).set_index("model")
display(validation_results)
chosen = validation_results["MAE"].idxmin()
chosen_features = feature_sets[chosen]
print("冻结方案：", chosen, chosen_features)


### 5. 冻结方案，最终评价

开发集为训练+验证；此后查看测试只用于报告。不会根据测试误差更换方案。

In [ ]:
# 选择完后，用训练+验证重新拟合；包括填补统计量。
development = pd.concat([train, val])
final_model = make_model().fit(development[chosen_features], development["G3"])
final_baseline = DummyRegressor(strategy="mean").fit(
    development[chosen_features], development["G3"])
test_pred = final_model.predict(test[chosen_features])
test_scores = pd.DataFrame({
    "baseline": metrics(test["G3"], final_baseline.predict(test[chosen_features])),
    "selected_linear": metrics(test["G3"], test_pred)}).T
display(test_scores)
# 评估后不再依据测试分数改特征、改参数或换随机种子。


In [ ]:
errors = pd.DataFrame({"actual": test["G3"], "predicted": test_pred}, index=test.index)
errors["residual"] = errors["actual"] - errors["predicted"]
errors["absolute_error"] = errors["residual"].abs()
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].scatter(errors["actual"], errors["predicted"], alpha=0.7)
lo = min(errors["actual"].min(), errors["predicted"].min())
hi = max(errors["actual"].max(), errors["predicted"].max())
axes[0].plot([lo,hi],[lo,hi],"r--")
axes[0].set(xlabel="Actual", ylabel="Predicted", title="Final held-out predictions")
axes[1].scatter(errors["predicted"], errors["residual"], alpha=0.7)
axes[1].axhline(0,color="r",linestyle="--")
axes[1].set(xlabel="Predicted", ylabel="Actual - predicted", title="Residuals")
fig.tight_layout()
plt.show()
display(errors.nlargest(5, "absolute_error"))


### 6. 结果分析与导出

下面的结论由实际执行结果生成；不能复制未经运行的预设分数。
模型系数仍是条件关联。保留极端误差与范围外预测，不裁剪以美化分数。

In [ ]:
selected = test_scores.loc["selected_linear"]
base = test_scores.loc["baseline"]
print(f"1. 训练 {len(train)}、验证 {len(val)}、测试 {len(test)} 条记录。")
print(f"2. 按验证 MAE 选择 {chosen}，最终特征为 {chosen_features}。")
print(f"3. 测试平均绝对误差 {selected['MAE']:.3f} 分（20 分制）。")
print(f"4. 相比均值基准，MAE 改善 {base['MAE']-selected['MAE']:.3f} 分；负值表示退步。")
print(f"5. 测试 RMSE {selected['RMSE']:.3f}，R² {selected['R2']:.3f}。")
print(f"6. 最大绝对误差 {errors['absolute_error'].max():.3f} 分，范围外预测 {((test_pred<0)|(test_pred>20)).sum()} 个。")
display(pd.Series(final_model.named_steps["regressor"].coef_, index=chosen_features, name="coefficient"))
output = ROOT / "assignments/bootcamp/week1/outputs"
output.mkdir(exist_ok=True)
test_scores.to_csv(output / "final_metrics.csv")
errors.to_csv(output / "final_predictions.csv", index_label="source_row")
fig.savefig(output / "final_diagnostics.png", dpi=150, bbox_inches="tight")

### 7. 局限和改进

- 样本规模有限，随机留出只能评估相似来源，不能代表所有学生。
- 不同学校或年份的关系可能改变，需要外部数据。
- 既往成绩已知是本项目假设；早期预测需去除未来字段。
- 零分记录可能包含多种实际情形，当前数据不能完整区分。
- 线性关系和档位等间距是假设，系数不能解释为因果影响。
- 下一步可以在新的实验方案中探索正则化或类别编码，再用新留出数据验证。

### 独立完成要求

复制本 Notebook 到自己的文件名，保留数据来源与拆分原则。自己编写质量报告、四张图的解释和五条结论。
Bonus：在训练集交叉验证中比较无 G1/G2 的场景，说明任务难度改变；不要用已看过的测试集宣称新方案泛化提高。